WARNING: Notebook was written during development. May not work as intended. Retained here only for completion and future reference.

In [ ]:
import functools
from typing import Tuple

import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt
import numpy as np

import crops.src.bc as _bc
import crops.src.contact as _contact
import crops.src.fe_cosserat as _fe
import crops.src.mesher as _mesher
import crops.src.solver as _solver
import crops.src.utils as _utils
import crops.src.viz as _viz

jax.config.update("jax_enable_x64", True)

### Simple wall sdf

In [ ]:
def build_half_space_sdf(
  wall_coord: float,
  extent_x: _utils.Extent,
  extent_y: _utils.Extent,
  num_grid_x: int = 41,
  num_grid_y: int = 121,
) -> Tuple[jnp.ndarray, jnp.ndarray, jnp.ndarray]:
  """Sample the signed distance field of a horizontal half-space onto a 2D grid.

  The obstacle occupies the half-space `y >= wall_coord`, so the signed distance of
  a query point is
                 phi(x, y) = wall_coord - y,

  which is positive below the wall (outside the obstacle), zero on its surface, and
  negative above (inside).

  Args:
    wall_coord: Scalar float [m]. The `y` coordinate of the wall surface. The
      obstacle occupies everything above it.
    extent_x: `Extent` [m] spanning the grid along the global x axis.
    extent_y: `Extent` [m] spanning the grid along the global y axis.
    num_grid_x: Number of grid nodes along x. Defaults to 41.
    num_grid_y: Number of grid nodes along y. Defaults to 121.

  Returns:
    A tuple `(sdf_grid, physical_origin, spacing)`. The grid has shape
      `(num_grid_x, num_grid_y)` [m], the origin has shape `(num_dim=2,)` [m] and
      locates grid node `[0, 0]`, and the spacing has shape `(num_dim=2,)` [m].
  """
  grid_x = jnp.linspace(extent_x.min, extent_x.max, num_grid_x)
  grid_y = jnp.linspace(extent_y.min, extent_y.max, num_grid_y)
  sdf_grid = wall_coord - jnp.einsum("x, y -> xy", jnp.ones(num_grid_x), grid_y)
  physical_origin = jnp.array([grid_x[0], grid_y[0]])
  spacing = jnp.array([grid_x[1] - grid_x[0], grid_y[1] - grid_y[0]])
  return sdf_grid, physical_origin, spacing

### Rod geom

In [ ]:
rod_length = 2.0

num_elems = 60
num_nodes = num_elems + 1

fiber_offset = 0.1  # c [m]
fiber_radius = 0.02

In [ ]:
# Four fibers at (+/-c, 0) and (0, +/-c).
offsets_local = np.array(
  [[fiber_offset, 0.0], [-fiber_offset, 0.0], [0.0, fiber_offset], [0.0, -fiber_offset]]
)
num_fibers = offsets_local.shape[0]
area_per_fiber = np.pi * fiber_radius**2

rod = _mesher.build_rod_geometry(
  analytical_curve=lambda k: jnp.array([k * rod_length, 0.0, 0.0]),
  parametric_coords=jnp.linspace(0.0, 1.0, num_nodes),
  fiber_offsets=jnp.asarray(np.tile(offsets_local, (num_elems, 1, 1))),
  fiber_areas=jnp.full((num_elems, num_fibers), area_per_fiber),
  fiber_mass_density=jnp.ones((num_elems, num_fibers)),
  fiber_centroid_moi=jnp.zeros((num_elems, num_fibers, 2)),
  seed_binormal=jnp.array([0.0, 0.0, 1.0]),
)

### BC

In [ ]:
num_load_steps = 10

clamped_dofs = (
  _utils.Translation.X,
  _utils.Translation.Y,
  _utils.Translation.Z,
  _utils.Rotation.THETA_X,
  _utils.Rotation.THETA_Y,
  _utils.Rotation.THETA_Z,
)
clamp = [
  _bc.DirichletBC(
    values=(dof, [np.array([0.0]) for _ in range(num_load_steps)]),
    node_targets=[np.array([0]) for _ in range(num_load_steps)],
    name=f"clamp_{dof}",
  )
  for dof in clamped_dofs
]

boundary_conditions = _bc.process_boundary_conditions(rod, clamp)

In [ ]:
remnant_flux = 0.1  # B_r [T], along the rod axis
applied_flux_max = 2.0e-2  # T, final value of the ramp

# B_r is per element per fiber, in the reference global frame: (e, f, d).
remnant_mag_flux = jnp.asarray(
  np.tile(np.array([remnant_flux, 0.0, 0.0]), (num_elems, num_fibers, 1))
)

### Material

In [ ]:
shear_modulus = 1.0e6  # G [Pa]
shear_moduli = shear_modulus * jnp.ones((num_elems, num_fibers))

### Solver

In [ ]:
nonlinear_settings = {
  "max_iter": 100,
  "threshold": 1e-6,
  "abs_threshold": 1e-6,
  "lam_min": 0.01,
  "line_search_max_iter": 15,
  "line_search_shrink": 0.5,
  "line_search_alpha_min": 1e-6,
  "line_search_armijo_c": 1e-4,
}

linear_settings = {
  "solver": _solver.LinearSolvers.SCIPY_SPARSE,
  "rtol": 1e-12,
}

nr_solver = _solver.NewtonRaphson(
  solver_type=_solver.NonlinearSolvers.ARMIJO_BACKTRACKING_NEWTON_RAPHSON,
  nonlinear_settings=nonlinear_settings,
  linear_settings=linear_settings,
)

In [ ]:
youngs_modulus = 3.0 * shear_modulus
area_total = num_fibers * area_per_fiber
moi_11 = np.sum(area_per_fiber * offsets_local[:, 0] ** 2)  # pairs with kappa_2

bending_stiffness = youngs_modulus * moi_11
shear_stiffness = shear_modulus * area_total

contact_radius = float(jnp.max(rod.contact_radii))
compliance_tip = (
  rod_length**3 / (3.0 * bending_stiffness) + rod_length / shear_stiffness
)

tip_force_max = (
  area_total * remnant_flux * applied_flux_max / _utils.PERMEABILITY_VACUUM
)
tip_deflection_free = tip_force_max * compliance_tip

# Place the effective centerline barrier at half of the unobstructed tip deflection,
# so the first half of the field ramp is contact free and the second half is not.
barrier_coord = 0.5 * tip_deflection_free
wall_coord = barrier_coord + contact_radius

print(f"contact radius (max over fibers)  = {contact_radius:.4f} m")
print(f"tip compliance                    = {compliance_tip:.6e} m/N")
print(f"free tip deflection at full field = {tip_deflection_free:.6f} m")
print(f"effective centerline barrier      = {barrier_coord:.6f} m")
print(f"wall surface coordinate           = {wall_coord:.6f} m")
print(
  f"beta at full field                = "
  f"{tip_force_max * rod_length**2 / bending_stiffness:.4f}"
)

### Problem setup

In [ ]:
elem_length = rod_length / num_elems
penalty_ratio = 1.0e3
wall_penalty_stiffness = penalty_ratio / (compliance_tip * elem_length)

sdf_grid, physical_origin, spacing = build_half_space_sdf(
  wall_coord=wall_coord,
  extent_x=_utils.Extent(min=-0.1, max=rod_length + 0.1),
  extent_y=_utils.Extent(min=-0.3, max=0.3),
)
obstacle = _contact.Obstacle(
  sdf_grid=sdf_grid,
  physical_origin=physical_origin,
  spacing=spacing,
  wall_penalty_stiffness=wall_penalty_stiffness,
  extrap_fill_value=10.0,  # far outside the grid means "no contact", not "inside"
)
problem_wall = _fe.CosseratProblem(
  rod=rod, bc=boundary_conditions, nr_solver=nr_solver, obstacle=obstacle
)

problem = _fe.CosseratProblem(
  rod=rod, bc=boundary_conditions, nr_solver=nr_solver, obstacle=None
)

### Inactive wall

In [ ]:
@functools.partial(jax.jit, static_argnames=("target_problem"))
def solve_at_field(
  target_problem: _fe.CosseratProblem, flux_magnitude: jnp.ndarray
) -> np.ndarray:
  """Solve to a given applied flux magnitude, ramping from zero."""
  field_ramp = jnp.einsum(
    "s, d -> sd",
    jnp.linspace(flux_magnitude / num_load_steps, flux_magnitude, num_load_steps),
    jnp.array([0.0, 1.0, 0.0]),
  )
  u_star, _ = target_problem.solve(
    u0=jnp.zeros(rod.num_dofs),
    shear_moduli=jnp.full((num_elems, num_fibers), shear_modulus),
    remnant_mag_flux=remnant_mag_flux,
    applied_mag_flux=field_ramp,
  )
  return u_star.reshape(num_nodes, rod.num_dofs_per_node)


small_field = 0.4 * applied_flux_max  # well below contact initiation
u_free = solve_at_field(problem, small_field)
u_wall = solve_at_field(problem_wall, small_field)
print(f"tip deflection, no obstacle   = {u_free[-1, 1]:.10e}")
print(f"tip deflection, wall inactive = {u_wall[-1, 1]:.10e}")
print(f"max |difference| over all dofs = {np.abs(u_wall - u_free).max():.3e}")

### Active wall

In [ ]:
flux_sweep = np.linspace(0.05, 1.0, 10) * applied_flux_max
tip_free, tip_walled = [], []

for flux in flux_sweep:
  tip_free.append(solve_at_field(problem, flux)[-1, 1])
  tip_walled.append(solve_at_field(problem_wall, flux)[-1, 1])
tip_free, tip_walled = np.array(tip_free), np.array(tip_walled)

force_sweep = area_total * remnant_flux * flux_sweep / _utils.PERMEABILITY_VACUUM
flux_initiation = (
  barrier_coord
  / compliance_tip
  * _utils.PERMEABILITY_VACUUM
  / (area_total * remnant_flux)
)
reaction_exact = np.maximum(force_sweep - barrier_coord / compliance_tip, 0.0)
reaction_penalty = (force_sweep * compliance_tip - tip_walled) / compliance_tip

In [ ]:
fig, (ax_tip, ax_force) = plt.subplots(1, 2, figsize=(11, 4))
ax_tip.plot(flux_sweep * 1e3, tip_free, "k--", label="no obstacle")
ax_tip.plot(flux_sweep * 1e3, tip_walled, "o-", ms=4, label="with wall")
ax_tip.axhline(barrier_coord, color="r", ls=":", label="barrier")
ax_tip.axvline(flux_initiation * 1e3, color="g", ls=":", label="predicted onset")
ax_tip.set_xlabel("$B_a$ [mT]")
ax_tip.set_ylabel("tip deflection [m]")
ax_tip.legend()

ax_force.plot(flux_sweep * 1e3, reaction_exact, "k-", lw=2, label="rigid-wall exact")
ax_force.plot(
  flux_sweep * 1e3, reaction_penalty, "o", ms=4, mfc="none", label="from penalty solve"
)
ax_force.set_xlabel("$B_a$ [mT]")
ax_force.set_ylabel("tip reaction [N]")
ax_force.legend()
fig.tight_layout()
plt.show()
print(f"predicted contact onset at B_a = {flux_initiation * 1e3:.4f} mT")

### Grad checks

In [ ]:
def tip_height_walled(remnant, applied_scalar):
  """Tip y-displacement with the wall active, as a function of the magnetic design."""
  field_ramp = jnp.einsum(
    "s, d -> sd",
    jnp.linspace(applied_scalar / num_load_steps, applied_scalar, num_load_steps),
    jnp.array([0.0, 1.0, 0.0]),
  )
  u_star, _ = problem_wall.solve(
    u0=jnp.zeros(rod.num_dofs),
    shear_moduli=jnp.full((num_elems, num_fibers), shear_modulus),
    remnant_mag_flux=remnant,
    applied_mag_flux=field_ramp,
  )
  return u_star.reshape(num_nodes, rod.num_dofs_per_node)[-1, 1]


# Check at a field WELL ABOVE onset, so contact is genuinely active and the tangent
# actually contains the contact contribution.
active_field = 0.9 * applied_flux_max
grad_remnant = jax.grad(tip_height_walled, argnums=0)(remnant_mag_flux, active_field)

step = 1e-7
probe = [(0, 0, 1), (num_elems // 2, 1, 1), (num_elems - 1, 0, 0)]
print(f"{'entry':>16} {'jax.grad':>16} {'central FD':>16} {'rel err':>11}")
for idx in probe:
  plus = remnant_mag_flux.at[idx].add(step)
  minus = remnant_mag_flux.at[idx].add(-step)
  fd = (tip_height_walled(plus, active_field)
        - tip_height_walled(minus, active_field)) / (2.0 * step)
  print(f"{str(idx):>16} {grad_remnant[idx]:16.8e} {fd:16.8e} "
        f"{abs(grad_remnant[idx] - fd) / max(abs(fd), 1e-30):11.2e}")